# StudentLife Bridge Model Analysis

This notebook tests whether StudentLife variables can support a **sleep-quality bridge** for the installation.

It uses short sections and plain language so the results are easier to scan.

**Important rule:** this notebook does **not** use `sleep_score` as evidence.

## Tested paths

A. **Screen time / phone use → sleep quality**

B. **Sleep quality → mood**

C. **Sleep quality → stress**

Each path uses:

- a baseline mean model;
- linear regression;
- random forest as a stronger comparison when enough rows exist.

A bridge is marked usable only when a tested model beats baseline on the held-out test set.

In [1]:
from pathlib import Path
import zipfile
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    import pyreadr
except Exception as exc:
    pyreadr = None
    print(f"pyreadr unavailable: {exc}")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

DATA_DIR = Path("data/studentlife_rds")
ZIP_PATH = DATA_DIR / "dataset_rds.zip"
TARGET_FILES = [
    "phonelock.Rds",
    "Sleep.Rds",
    "Mood.Rds",
    "PerceivedStressScale.Rds",
    "Stress.Rds",
    "PAM.Rds",
]

print("Data directory:", DATA_DIR.resolve())
print("Archive exists:", ZIP_PATH.exists())

Data directory: /workspace/Sleep-datasets-official/data/studentlife_rds
Archive exists: False


In [2]:
# Extract only when the archive is present and the target RDS files are not already available.
DATA_DIR.mkdir(parents=True, exist_ok=True)
found_before = {p.name: p for p in DATA_DIR.rglob("*.Rds")}
if ZIP_PATH.exists() and not all(name in found_before for name in TARGET_FILES[:5]):
    print("Extracting dataset_rds.zip ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR)
else:
    print("No extraction needed, or archive is missing.")

found = {}
for name in TARGET_FILES:
    matches = list(DATA_DIR.rglob(name))
    found[name] = matches[0] if matches else None

pd.DataFrame(
    [{"file": name, "found": path is not None, "path": str(path) if path else ""} for name, path in found.items()]
)

No extraction needed, or archive is missing.


,file,found,path
0,phonelock.Rds,False,
1,Sleep.Rds,False,
2,Mood.Rds,False,
3,PerceivedStressScale.Rds,False,
4,Stress.Rds,False,
5,PAM.Rds,False,


In [3]:
def read_rds(path):
    if path is None:
        return None
    if pyreadr is None:
        raise RuntimeError("pyreadr is required to read RDS files")
    result = pyreadr.read_r(str(path))
    if not result:
        return None
    # Most pyreadr RDS reads use None as the key.
    return next(iter(result.values()))

frames = {}
for name, path in found.items():
    if path is None:
        frames[name] = None
        continue
    df = read_rds(path)
    frames[name] = df
    print(f"{name}: shape={df.shape}")
    print(list(df.columns)[:30])

In [4]:
def normalize_studentlife(df):
    """Create conservative join keys and numeric candidate columns.

    This function avoids invented connections. It only uses variables present in the RDS files.
    """
    if df is None or df.empty:
        return df
    out = df.copy()
    # Common StudentLife columns vary by exported table.
    uid_candidates = [c for c in out.columns if c.lower() in {"uid", "user", "participant", "id", "subject"}]
    if uid_candidates:
        out["_uid"] = out[uid_candidates[0]].astype(str)
    else:
        out["_uid"] = "all"

    time_cols = [c for c in out.columns if any(k in c.lower() for k in ["time", "date", "day", "timestamp"])]
    parsed = None
    for c in time_cols:
        s = out[c]
        if pd.api.types.is_numeric_dtype(s):
            med = pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan
            unit = "s" if pd.notna(med) and med > 10_000_000 else None
            dt = pd.to_datetime(s, errors="coerce", unit=unit)
        else:
            dt = pd.to_datetime(s, errors="coerce")
        if dt.notna().sum() >= max(3, len(out) * 0.2):
            parsed = dt
            break
    if parsed is not None:
        out["_date"] = parsed.dt.date.astype(str)
    else:
        out["_date"] = "unknown"
    return out

norm = {name: normalize_studentlife(df) for name, df in frames.items()}

for name, df in norm.items():
    if df is not None:
        print(name, "join columns present:", {c: c in df.columns for c in ["_uid", "_date"]})

In [5]:
def numeric_cols(df, exclude=("_uid", "_date")):
    if df is None:
        return []
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def pick_cols(df, keywords, max_cols=8):
    cols = []
    if df is None:
        return cols
    for c in numeric_cols(df):
        lc = c.lower()
        if any(k in lc for k in keywords):
            cols.append(c)
    return cols[:max_cols]

def daily_numeric(df, cols, prefix):
    if df is None or not cols:
        return None
    keep = ["_uid", "_date"] + cols
    agg = df[keep].groupby(["_uid", "_date"], as_index=False).mean(numeric_only=True)
    return agg.rename(columns={c: f"{prefix}{c}" for c in cols})

def evaluate_path(name, left, right, predictors, target):
    if left is None or right is None or not predictors or target is None:
        return {"path": name, "status": "not tested", "reason": "missing file, predictor, or target"}
    data = pd.merge(left, right, on=["_uid", "_date"], how="inner")
    cols = predictors + [target]
    data = data.dropna(subset=cols)
    if len(data) < 20:
        return {"path": name, "status": "not tested", "reason": f"not enough joined rows ({len(data)})"}
    X = data[predictors]
    y = data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    baseline_pred = np.repeat(y_train.mean(), len(y_test))
    baseline = {
        "r2": r2_score(y_test, baseline_pred),
        "mae": mean_absolute_error(y_test, baseline_pred),
        "rmse": mean_squared_error(y_test, baseline_pred) ** 0.5,
    }
    linear = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LinearRegression())])
    linear.fit(X_train, y_train)
    lp = linear.predict(X_test)
    linear_metrics = {"r2": r2_score(y_test, lp), "mae": mean_absolute_error(y_test, lp), "rmse": mean_squared_error(y_test, lp) ** 0.5}
    rf_metrics = None
    if len(data) >= 40:
        rf = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3))])
        rf.fit(X_train, y_train)
        rp = rf.predict(X_test)
        rf_metrics = {"r2": r2_score(y_test, rp), "mae": mean_absolute_error(y_test, rp), "rmse": mean_squared_error(y_test, rp) ** 0.5}
    usable = (linear_metrics["rmse"] < baseline["rmse"] and linear_metrics["mae"] < baseline["mae"]) or (rf_metrics and rf_metrics["rmse"] < baseline["rmse"] and rf_metrics["mae"] < baseline["mae"])
    return {"path": name, "status": "tested", "rows_used": len(data), "target": target, "predictors": predictors, "baseline": baseline, "linear": linear_metrics, "random_forest": rf_metrics, "usable": bool(usable)}

In [6]:
sleep = norm.get("Sleep.Rds")
phone = norm.get("phonelock.Rds")
mood = norm.get("Mood.Rds")
pss = norm.get("PerceivedStressScale.Rds")
stress = norm.get("Stress.Rds")

sleep_quality_cols = pick_cols(sleep, ["rate", "quality", "hour", "sleep", "duration"])
phone_cols = pick_cols(phone, ["lock", "unlock", "duration", "screen", "phone"])
mood_cols = pick_cols(mood, ["happy", "sad", "mood", "score"])
stress_cols = pick_cols(pss, ["pss", "stress", "score"]) or pick_cols(stress, ["stress", "level", "score"])

print("Sleep candidates:", sleep_quality_cols)
print("Phone candidates:", phone_cols)
print("Mood candidates:", mood_cols)
print("Stress candidates:", stress_cols)

Sleep candidates: []
Phone candidates: []
Mood candidates: []
Stress candidates: []


In [7]:
sleep_daily = daily_numeric(sleep, sleep_quality_cols, "sleep__")
phone_daily = daily_numeric(phone, phone_cols, "phone__")
mood_daily = daily_numeric(mood, mood_cols, "mood__")
pss_daily = daily_numeric(pss, stress_cols, "stress__") if pss is not None else None
stress_daily = daily_numeric(stress, stress_cols, "stress__") if pss_daily is None else None
stress_source_daily = pss_daily if pss_daily is not None else stress_daily

sleep_targets = [f"sleep__{c}" for c in sleep_quality_cols]
phone_predictors = [f"phone__{c}" for c in phone_cols]
mood_targets = [f"mood__{c}" for c in mood_cols]
stress_targets = [f"stress__{c}" for c in stress_cols]

results = []
results.append(evaluate_path("A. screen time / phone use -> sleep quality", phone_daily, sleep_daily, phone_predictors, sleep_targets[0] if sleep_targets else None))
results.append(evaluate_path("B. sleep quality -> mood", sleep_daily, mood_daily, sleep_targets, mood_targets[0] if mood_targets else None))
results.append(evaluate_path("C. sleep quality -> stress", sleep_daily, stress_source_daily, sleep_targets, stress_targets[0] if stress_targets else None))

print(json.dumps(results, indent=2))

[
  {
    "path": "A. screen time / phone use -> sleep quality",
    "status": "not tested",
    "reason": "missing file, predictor, or target"
  },
  {
    "path": "B. sleep quality -> mood",
    "status": "not tested",
    "reason": "missing file, predictor, or target"
  },
  {
    "path": "C. sleep quality -> stress",
    "status": "not tested",
    "reason": "missing file, predictor, or target"
  }
]


## Notebook decision rule

A path is **usable** only if a tested model improves over the baseline mean model on the test data.

If the files are missing, the path is **not tested** and should not be used as evidence.